In [10]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)
    
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)

seed = 42


sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)
matplotlib 3.10.1
numpy 2.2.4
pandas 2.2.3
sklearn 1.6.1
torch 2.7.0+cpu
cpu


# VGG Net
相比AlexNet，使用了可重复使用的VGG块和可供调节的超参数
VGG是更大更深的AlexNet提出了VGG块，更加规整
VGG使用可以重复使用的卷积快来构建深度卷积神经网络
不同的卷积块个数和超参数可以得到不同的复杂度的变种

In [3]:
class VGGNetBlock(nn.Module):
     def __init__(self, num_conv, in_channels, out_channels):
         super().__init__()
         self.model = nn.Sequential()
         for i in range(num_conv):
             self.model.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
             self.model.append(nn.ReLU(inplace=True))
             in_channels = out_channels
         self.model.append(nn.MaxPool2d(kernel_size=2, stride=1))
     
     def forward(self, x):
         return self.model(x)

In [5]:
conv_arch = ((1, 64), (1, 128), (2, 256), (2, 512), (2, 512)) # 前面是卷积数目，后面是输出通道数
# 这里设计了5层卷积块因为输入的图像尺寸是(224,224),在经过5次VGG_Block的最大值池化后会使得图像缩减为(7,7)不可以再继续除2

class VGGNet(nn.Module):
    def __init__(self, conv_arch, class_num):
        super().__init__()
        self.model = nn.Sequential()
        self.in_channels = 3
        self.out_channels = 3
        for idx, value in enumerate(conv_arch):
            self.out_channels = value[1]
            self.model.append(VGGNetBlock(value[0], in_channels=self.in_channels, out_channels=self.out_channels))
            self.in_channels = self.out_channels
        nn.Flatten()
        nn.Linear(self.out_channels * 7 * 7, 4096)
        nn.ReLU()
        nn.Dropout(0.5)
        nn.Linear(4096, 4096)
        nn.Dropout(0.5)
        nn.Linear(4096, class_num)
    def forward(self, x): 
        return self.model(x)

In [6]:
x = torch.rand((1, 3, 224, 224))
model = VGGNet(conv_arch, class_num=10)
model.to(device)
model(x)

tensor([[[[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          ...,
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00]],

         [[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          ...,
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000

In [8]:
for key, value in VGGNet(conv_arch, 10).named_parameters():   
     print(f"{key:^50}paramerters num: {np.prod(value.shape)}")

              model.0.model.0.weight              paramerters num: 1728
               model.0.model.0.bias               paramerters num: 64
              model.1.model.0.weight              paramerters num: 73728
               model.1.model.0.bias               paramerters num: 128
              model.2.model.0.weight              paramerters num: 294912
               model.2.model.0.bias               paramerters num: 256
              model.2.model.2.weight              paramerters num: 589824
               model.2.model.2.bias               paramerters num: 256
              model.3.model.0.weight              paramerters num: 1179648
               model.3.model.0.bias               paramerters num: 512
              model.3.model.2.weight              paramerters num: 2359296
               model.3.model.2.bias               paramerters num: 512
              model.4.model.0.weight              paramerters num: 2359296
               model.4.model.0.bias               paramer

In [9]:
#模型总参数量
total_params = sum(p.numel() for p in VGGNet(conv_arch, 10).parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params}")

Total trainable parameters: 9220480
